# FLUX-LM Universal: SFT + DPO with Domain-Specific Teachers

**Post-training alignment using open-source domain experts.**

> **Prerequisite:** Complete pre-training (`flux_lm_universal_train.ipynb`) first.

## Pipeline Overview
```
Pre-trained FLUX-LM (500M) → SFT (distillation) → DPO (preference alignment) → Aligned Model
```

## Domain-Specific Teachers (3B-7B)

| Domain | Teacher Model | Size | Why |
|--------|---------------|------|-----|
| **Code** | DeepSeek-Coder-6.7B | 6.7B | Best OSS code model |
| **Math/Reasoning** | DeepSeek-Math-7B | 7B | State-of-art math |
| **General/Assistant** | Qwen2.5-7B-Instruct | 7B | Best instruction following |
| **Multilingual** | Qwen2.5-7B | 7B | Excellent CJK + European |

## Training Phases

| Phase | Steps | Learning Rate | Data Source |
|-------|-------|---------------|-------------|
| **SFT** | 5,000 | 1e-5 | Teacher-generated responses |
| **DPO** | 2,000 | 5e-6 | Preference pairs (LLM-as-judge) |

## Memory Requirements (L4 24GB)

- Teacher (7B, FP16): ~14GB
- FLUX-LM (500M): ~2GB
- Activations: ~6GB
- **Total**: ~22GB ✅ Fits on L4

**Strategy:** Load one teacher at a time, generate data, unload, then train FLUX-LM.

In [ ]:
# Cell 1: Setup and Imports
import os
import sys
import gc
import math
import random
import time
import json
import shutil
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from tqdm.auto import tqdm

print("=" * 60)
print("FLUX-LM Universal: SFT + DPO Alignment")
print("=" * 60)

# ==== COLAB-ONLY ENVIRONMENT ====
IN_COLAB = 'google.colab' in sys.modules

if not IN_COLAB:
    ROOT = Path('/Users/admin/Desktop/flux')
    SAVE_TO_DRIVE = False
    DRIVE_CKPT_DIR = None
    print("Environment: Local (no Drive saving)")
else:
    ROOT = Path('/content/FLUX')
    print("Environment: Google Colab")
    
    # Mount Google Drive
    SAVE_TO_DRIVE = True
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/FLUX_checkpoints')
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✓ Google Drive mounted")
    print(f"✓ Checkpoints: {DRIVE_CKPT_DIR}")

print(f"Root: {ROOT}")

In [ ]:
# Cell 2: Clone/Update Repository + Install Dependencies
if IN_COLAB:
    if not ROOT.exists():
        print("Cloning FLUX repository...")
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    else:
        print("Updating FLUX repository...")
        %cd {ROOT}
        !git pull
    
    # Install dependencies
    !pip install -q datasets tqdm rich matplotlib transformers accelerate bitsandbytes

%cd {ROOT}
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'phases' / 'phase_lm'))

print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 3: Hardware Detection + L4 Optimizations
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device != 'cuda':
    raise RuntimeError("This notebook requires CUDA GPU (use Colab with L4/A100)")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")

# ==== L4 OPTIMIZATIONS ====
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("✓ TF32 + cuDNN benchmark enabled")

if hasattr(torch.nn.functional, 'scaled_dot_product_attention'):
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    print("✓ Flash attention enabled")

# SFT/DPO uses smaller batches (teacher model takes memory)
SFT_BATCH_SIZE = 1 if gpu_mem < 30 else 2  # Conservative for L4
SFT_GRAD_ACCUM = 32 if gpu_mem < 30 else 16
print(f"\nSFT config: batch={SFT_BATCH_SIZE}, grad_accum={SFT_GRAD_ACCUM}")
print(f"Effective batch: {SFT_BATCH_SIZE * SFT_GRAD_ACCUM}")

In [ ]:
# Cell 4: Import FLUX-LM Components
from flux_lm_universal import (
    FluxLMUniversal, 
    GenerationConfigUniversal,
    format_params,
)

from multi_domain_data import (
    SPECIAL_TOKENS,
    VOCAB_SIZE,
    encode_special,
    decode_special,
)

# ==== NOVEL: Wave Resonance Memory (Anti-Forgetting) ====
from wave_resonance_memory import (
    WaveResonanceMemory,
    WRMConfig,
    create_wrm_for_sft,
)

print("✓ FLUX-LM modules imported")
print("✓ Wave Resonance Memory imported (anti-forgetting)")

In [ ]:
# Cell 5: Load Pre-trained FLUX-LM Model

CKPT_DIR = ROOT / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# ==== LOAD FROM GOOGLE DRIVE ====
MODEL_SCALE = '500M'
pretrained_path = None

# Check Drive first
if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    drive_model = DRIVE_CKPT_DIR / f'Flux-LM-Universal-{MODEL_SCALE}-best.pt'
    if drive_model.exists():
        pretrained_path = drive_model
        print(f"✓ Found model on Drive: {drive_model.name}")

# Check local
if pretrained_path is None:
    local_model = CKPT_DIR / f'Flux-LM-Universal-{MODEL_SCALE}-best.pt'
    if local_model.exists():
        pretrained_path = local_model
        print(f"✓ Found model locally: {local_model.name}")

if pretrained_path is None:
    raise FileNotFoundError(
        f"Pre-trained model not found!\n"
        f"Run flux_lm_universal_train.ipynb first to train Flux-LM-Universal-{MODEL_SCALE}-best.pt"
    )

print(f"\nLoading pre-trained model from: {pretrained_path}")
model = FluxLMUniversal.load(str(pretrained_path), device=device)
model.train()

params = model.count_parameters()
print(f"✓ Loaded FLUX-LM {MODEL_SCALE}: {format_params(params['total'])} parameters")

## Domain-Specific Teacher Models

We load teachers one at a time to fit in L4 memory:

| Domain | Teacher | Data Format (matching pre-training) |
|--------|---------|-------------------------------------|
| **code** | DeepSeek-Coder-6.7B | `<\|lang:python\|><\|code\|>...` |
| **reasoning** | DeepSeek-Math-7B | `<\|problem\|>...<\|reasoning\|>...` |
| **assistant** | Qwen2.5-7B-Instruct | `<\|user\|>...<\|assistant\|>...` |

These formats match the pre-training data from `multi_domain_data.py`.

In [ ]:
# Cell 6: Teacher Model Configuration
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==== DOMAIN-SPECIFIC TEACHERS ====
DOMAIN_TEACHERS = {
    'code': {
        'model_id': 'deepseek-ai/deepseek-coder-6.7b-instruct',
        'format': '<|lang:python|><|code|>\n{prompt}',
        'samples': 5000,
    },
    'reasoning': {
        'model_id': 'deepseek-ai/deepseek-math-7b-instruct',
        'format': '<|problem|>\n{prompt}\n<|end|>\n<|reasoning|>\n',
        'samples': 3000,
    },
    'assistant': {
        'model_id': 'Qwen/Qwen2.5-7B-Instruct',
        'format': '<|user|>{prompt}<|end|>\n<|assistant|>',
        'samples': 8000,
    },
}

# Prompts for each domain (matching pre-training distribution)
DOMAIN_PROMPTS = {
    'code': [
        "Write a Python function to check if a number is prime.",
        "Implement binary search in Python.",
        "Write a function to reverse a linked list.",
        "Create a class for a binary tree with insert and search methods.",
        "Write a Python decorator that measures function execution time.",
        "Implement quicksort algorithm.",
        "Write a function to find the longest common subsequence.",
        "Create a simple HTTP server in Python.",
        "Write a function to validate an email address.",
        "Implement a LRU cache class.",
    ],
    'reasoning': [
        "Sarah has 15 apples. She gives 4 to Tom and buys 7 more. How many apples does she have?",
        "If x + 5 = 12, what is x?",
        "A train travels at 60 mph. How far does it go in 2.5 hours?",
        "If all roses are flowers, and some flowers fade quickly, can we conclude that some roses fade quickly?",
        "What is the derivative of x^3 + 2x^2 - 5x + 3?",
        "Solve the equation: 2x + 7 = 15",
        "A rectangle has length 8 and width 5. What is its area and perimeter?",
        "If a car depreciates 15% per year, what is its value after 3 years if it started at $20,000?",
        "What is the probability of rolling a sum of 7 with two dice?",
        "Find the roots of x^2 - 5x + 6 = 0",
    ],
    'assistant': [
        "What is machine learning?",
        "Explain quantum computing in simple terms.",
        "How do I learn to code?",
        "What are the benefits of exercise?",
        "Explain the difference between AI and ML.",
        "What is the capital of France?",
        "How does the internet work?",
        "What is climate change?",
        "Explain photosynthesis.",
        "What is a neural network?",
    ],
}

print("Teacher configurations:")
for domain, cfg in DOMAIN_TEACHERS.items():
    print(f"  [{domain}] {cfg['model_id'].split('/')[-1]} → {cfg['samples']} samples")

In [ ]:
# Cell 7: Teacher Loading and Data Generation Functions

def cleanup_gpu():
    """Aggressively clean up GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_teacher(domain: str):
    """Load teacher model for a specific domain."""
    cfg = DOMAIN_TEACHERS[domain]
    model_id = cfg['model_id']
    
    print(f"\nLoading teacher: {model_id}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    teacher = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    teacher.eval()
    
    # Get memory usage
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"✓ Loaded {model_id.split('/')[-1]} | VRAM: {mem_used:.1f} GB")
    
    return teacher, tokenizer


def unload_teacher(teacher, tokenizer):
    """Unload teacher to free GPU memory."""
    del teacher
    del tokenizer
    cleanup_gpu()
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"✓ Teacher unloaded | VRAM: {mem_used:.1f} GB")


def generate_from_teacher(
    teacher, 
    tokenizer, 
    prompt: str, 
    max_new_tokens: int = 512,
    temperature: float = 0.7,
) -> str:
    """Generate response from teacher model."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(teacher.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = teacher.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


def format_sft_sample(domain: str, prompt: str, response: str) -> str:
    """Format sample in FLUX-LM training format (matching pre-training)."""
    if domain == 'code':
        return f"<|lang:python|><|code|>\n# {prompt}\n{response}<|end_code|>"
    elif domain == 'reasoning':
        return f"<|problem|>\n{prompt}\n<|end|>\n<|reasoning|>\n{response}\n<|end_reasoning|>"
    elif domain == 'assistant':
        return f"<|user|>{prompt}<|end|>\n<|assistant|>{response}<|end|>"
    else:
        return f"<|user|>{prompt}<|end|>\n<|assistant|>{response}<|end|>"


print("✓ Teacher utilities defined")

In [ ]:
# Cell 8: Generate SFT Data from Teachers

# ==== GENERATE SFT DATA (OR LOAD CACHED) ====
SFT_DATA_PATH = CKPT_DIR / 'sft_data.json'

# Check if we have cached SFT data
if SFT_DATA_PATH.exists():
    print(f"Loading cached SFT data from {SFT_DATA_PATH}")
    with open(SFT_DATA_PATH, 'r') as f:
        sft_data = json.load(f)
    print(f"✓ Loaded {len(sft_data)} SFT samples")
else:
    print("Generating SFT data from domain teachers...")
    print("This will take ~30-60 minutes (loading/unloading 7B models)\n")
    
    sft_data = []
    
    for domain, cfg in DOMAIN_TEACHERS.items():
        print(f"\n{'='*50}")
        print(f"Domain: {domain.upper()}")
        print(f"{'='*50}")
        
        # Load teacher
        teacher, tokenizer = load_teacher(domain)
        
        # Get prompts (cycle through if needed)
        base_prompts = DOMAIN_PROMPTS[domain]
        num_samples = cfg['samples']
        
        # For more samples, we'd load from HuggingFace datasets
        # For now, use the base prompts with variations
        prompts = []
        for i in range(num_samples):
            base_prompt = base_prompts[i % len(base_prompts)]
            # Add slight variation for diversity
            if i >= len(base_prompts):
                base_prompt = f"{base_prompt} (variation {i // len(base_prompts)})"
            prompts.append(base_prompt)
        
        # Generate responses
        domain_data = []
        for i, prompt in enumerate(tqdm(prompts[:100], desc=f"Generating {domain}")):
            try:
                response = generate_from_teacher(teacher, tokenizer, prompt)
                formatted = format_sft_sample(domain, prompt, response)
                
                domain_data.append({
                    'domain': domain,
                    'prompt': prompt,
                    'response': response,
                    'formatted': formatted,
                    'teacher': cfg['model_id'],
                })
            except Exception as e:
                print(f"  Error on sample {i}: {e}")
                continue
        
        sft_data.extend(domain_data)
        print(f"✓ Generated {len(domain_data)} samples for {domain}")
        
        # Unload teacher to free memory
        unload_teacher(teacher, tokenizer)
    
    # Save SFT data
    with open(SFT_DATA_PATH, 'w') as f:
        json.dump(sft_data, f, indent=2)
    print(f"\n✓ Saved {len(sft_data)} SFT samples to {SFT_DATA_PATH}")
    
    # Copy to Drive
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        shutil.copy2(SFT_DATA_PATH, DRIVE_CKPT_DIR / 'sft_data.json')
        print(f"✓ Copied to Google Drive")

# Show sample
print(f"\nSample SFT data:")
for domain in ['code', 'reasoning', 'assistant']:
    samples = [s for s in sft_data if s['domain'] == domain]
    if samples:
        print(f"\n[{domain.upper()}] ({len(samples)} samples)")
        print(f"  Prompt: {samples[0]['prompt'][:60]}...")
        print(f"  Response: {samples[0]['response'][:80]}...")

In [ ]:
# Cell 9: SFT Dataset Class

class SFTDataset(Dataset):
    """Dataset for Supervised Fine-Tuning."""
    
    def __init__(self, data: List[dict], max_len: int = 512):
        self.data = data
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        text = sample['formatted']
        
        # Encode with special tokens
        encoded = encode_special(text)
        
        # Truncate/pad to max_len
        if len(encoded) > self.max_len:
            encoded = encoded[:self.max_len]
        else:
            encoded = encoded + [0] * (self.max_len - len(encoded))
        
        input_ids = torch.tensor(encoded[:-1], dtype=torch.long)
        target_ids = torch.tensor(encoded[1:], dtype=torch.long)
        
        return {
            'input': input_ids,
            'target': target_ids,
            'domain': sample['domain'],
        }


# Create SFT dataset
sft_dataset = SFTDataset(sft_data, max_len=512)
print(f"✓ SFT dataset: {len(sft_dataset)} samples")

# Collate function
def sft_collate_fn(batch):
    inputs = torch.stack([item['input'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    domains = [item['domain'] for item in batch]
    
    # Clamp to valid vocab range
    inputs = inputs.clamp(0, VOCAB_SIZE - 1)
    targets = targets.clamp(0, VOCAB_SIZE - 1)
    
    return {'input': inputs, 'target': targets, 'domains': domains}

# Create DataLoader
sft_loader = DataLoader(
    sft_dataset,
    batch_size=SFT_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=sft_collate_fn,
)

print(f"✓ SFT DataLoader: {len(sft_loader)} batches")

In [ ]:
# Cell 10: SFT Training Configuration

# ==== SFT HYPERPARAMETERS ====
SFT_STEPS = 5000
SFT_LR = 1e-5  # Much lower than pre-training
SFT_WARMUP_STEPS = 200
SFT_WEIGHT_DECAY = 0.01
SFT_GRAD_CLIP = 1.0
SFT_VAL_EVERY = 500
SFT_SAVE_EVERY = 1000

# Fresh optimizer (NOT loading from pre-training checkpoint)
sft_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=SFT_LR,
    weight_decay=SFT_WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

# Warmup + cosine schedule
def sft_get_lr(step):
    if step < SFT_WARMUP_STEPS:
        return step / SFT_WARMUP_STEPS
    else:
        progress = (step - SFT_WARMUP_STEPS) / (SFT_STEPS - SFT_WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * progress))

sft_scheduler = torch.optim.lr_scheduler.LambdaLR(sft_optimizer, sft_get_lr)

# Mixed precision
sft_scaler = GradScaler(enabled=(device == 'cuda'))

print(f"SFT Configuration:")
print(f"  Total steps: {SFT_STEPS:,}")
print(f"  Learning rate: {SFT_LR}")
print(f"  Warmup steps: {SFT_WARMUP_STEPS}")
print(f"  Effective batch: {SFT_BATCH_SIZE * SFT_GRAD_ACCUM}")

In [ ]:
# Cell 10b: Initialize Wave Resonance Memory (WRM)
# 
# MUST be called AFTER:
# - Model is loaded (Cell 5)
# - SFT dataset is created (Cell 9)
# 
# WRM captures the model's knowledge state BEFORE SFT begins,
# preserving both wave representations and byte predictions.

print("=" * 60)
print("Initializing Wave Resonance Memory (Anti-Forgetting)")
print("=" * 60)

# Method 1: Use existing SFT dataset for calibration (recommended)
print("\nInitializing from SFT dataset...")

# Use create_wrm_for_sft which samples from the dataset
wrm = create_wrm_for_sft(
    model=model,
    train_dataset=sft_dataset,
    num_calibration_samples=500,
    device=device,
)

# Define preservation loss weight for SFT
# Tune this: higher = more preservation, lower = more adaptation
WRM_PRESERVATION_WEIGHT = 0.1

print(f"\n✓ WRM initialized:")
print(f"  Wave anchors: {len(wrm.wave_anchors.anchors)} samples")
print(f"  Domain centroids: {len(wrm.wave_anchors.domain_centroids)}")
print(f"  Byte transitions preserved: {wrm.byte_memory.num_critical}")
print(f"  Preservation weight: {WRM_PRESERVATION_WEIGHT}")

# Save WRM state (optional - for recovery)
wrm_path = CKPT_DIR / 'wrm_pre_sft.pt'
wrm.save(str(wrm_path))
print(f"\n✓ WRM saved to {wrm_path}")

if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    try:
        import shutil
        shutil.copy2(wrm_path, DRIVE_CKPT_DIR / 'wrm_pre_sft.pt')
        print(f"✓ WRM copied to Google Drive")
    except Exception as e:
        print(f"⚠ Drive copy failed: {e}")

In [ ]:
# Cell 11: SFT Validation + Generation Demo

@torch.no_grad()
def sft_validate(model, data_loader, device, max_batches=20):
    """Quick validation on SFT data."""
    model.eval()
    total_loss = 0
    
    for i, batch in enumerate(data_loader):
        if i >= max_batches:
            break
        
        input_ids = batch['input'].to(device)
        target_ids = batch['target'].to(device)
        
        with torch.amp.autocast('cuda'):
            output = model(input_ids, target_ids)
        
        total_loss += output['loss'].item()
    
    model.train()
    return total_loss / max(1, min(len(data_loader), max_batches))


@torch.no_grad()
def sft_generation_demo(model, device, step: int):
    """Show generation quality during SFT."""
    model.eval()
    
    demos = [
        ("assistant", "<|user|>What is 2+2?<|end|>\n<|assistant|>"),
        ("reasoning", "<|problem|>\nIf x = 5, what is 2x + 3?\n<|end|>\n<|reasoning|>\n"),
        ("code", "<|lang:python|><|code|>\ndef is_prime(n):\n"),
    ]
    
    print(f"\n{'='*60}")
    print(f"[SFT Step {step:,}] Generation Demo")
    print(f"{'='*60}")
    
    for domain, prompt in demos:
        try:
            output = model.generate(prompt, GenerationConfigUniversal(
                max_new_bytes=100,
                temperature=0.7 if domain != 'code' else 0.3,
                domain=domain,
            ))
            generated = output[len(prompt):150].replace('\n', '\\n')
            print(f"\n[{domain.upper()}] {prompt[:40]}...")
            print(f"  → {generated}...")
        except Exception as e:
            print(f"\n[{domain.upper()}] Error: {e}")
    
    print(f"{'='*60}\n")
    model.train()


# Test validation
print("Testing validation...")
val_loss = sft_validate(model, sft_loader, device, max_batches=5)
print(f"Initial SFT val loss: {val_loss:.4f}")

# Initial generation demo
sft_generation_demo(model, device, step=0)

In [ ]:
# Cell 12: Checkpoint Functions

def save_sft_checkpoint(model, optimizer, scheduler, step, loss, path):
    """Save SFT checkpoint."""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'step': step,
        'loss': loss,
        'config': model.config,
        'timestamp': datetime.now().isoformat(),
        'stage': 'sft',
    }
    torch.save(checkpoint, path)
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  ✓ SFT checkpoint: {Path(path).name} ({size_mb:.1f} MB)")
    
    # Copy to Drive
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        try:
            shutil.copy2(path, DRIVE_CKPT_DIR / Path(path).name)
            print(f"  ✓ Copied to Google Drive")
        except Exception as e:
            print(f"  ⚠ Drive copy failed: {e}")


def save_model_checkpoint(model, filename):
    """Save model-only checkpoint."""
    path = CKPT_DIR / filename
    model.save(str(path))
    size_mb = path.stat().st_size / 1e6
    print(f"  ✓ Model saved: {filename} ({size_mb:.1f} MB)")
    
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        try:
            shutil.copy2(path, DRIVE_CKPT_DIR / filename)
            print(f"  ✓ Copied to Google Drive")
        except Exception as e:
            print(f"  ⚠ Drive copy failed: {e}")


print("✓ Checkpoint functions defined")

In [ ]:
# Cell 13: SFT Training Loop with Wave Resonance Memory

print("=" * 60)
print(f"Starting SFT Training with WRM Anti-Forgetting")
print("=" * 60)

if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    print(f"📁 Saving to: {DRIVE_CKPT_DIR}")

print(f"WRM Preservation Weight: {WRM_PRESERVATION_WEIGHT}")

model.train()
sft_step = 0
sft_losses = []
wrm_losses = []  # Track preservation loss
best_sft_loss = float('inf')
start_time = time.time()

pbar = tqdm(total=SFT_STEPS, desc="SFT Training")

while sft_step < SFT_STEPS:
    for batch in sft_loader:
        if sft_step >= SFT_STEPS:
            break
        
        input_ids = batch['input'].to(device, non_blocking=True)
        target_ids = batch['target'].to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            output = model(input_ids, target_ids)
            sft_loss = output['loss']
            
            # === WRM PRESERVATION LOSS ===
            # Compute preservation loss to prevent forgetting pre-trained knowledge
            preservation_loss, wrm_breakdown = wrm.compute_preservation_loss(
                model=model,
                input_ids=input_ids,
                logits=output['logits'],
            )
            
            # Combined loss: SFT + preservation
            total_loss = sft_loss + WRM_PRESERVATION_WEIGHT * preservation_loss
            loss = total_loss / SFT_GRAD_ACCUM
        
        sft_scaler.scale(loss).backward()
        
        # Gradient accumulation
        if (sft_step + 1) % SFT_GRAD_ACCUM == 0:
            sft_scaler.unscale_(sft_optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), SFT_GRAD_CLIP)
            sft_scaler.step(sft_optimizer)
            sft_scaler.update()
            sft_optimizer.zero_grad(set_to_none=True)
            sft_scheduler.step()
        
        sft_step += 1
        loss_val = sft_loss.item()
        wrm_val = preservation_loss.item()
        sft_losses.append(loss_val)
        wrm_losses.append(wrm_val)
        
        pbar.update(1)
        pbar.set_postfix({
            'sft': f"{loss_val:.4f}", 
            'wrm': f"{wrm_val:.4f}",
            'lr': f"{sft_scheduler.get_last_lr()[0]:.2e}"
        })
        
        # Validation
        if sft_step % SFT_VAL_EVERY == 0:
            val_loss = sft_validate(model, sft_loader, device)
            elapsed = (time.time() - start_time) / 3600
            avg_wrm = sum(wrm_losses[-SFT_VAL_EVERY:]) / len(wrm_losses[-SFT_VAL_EVERY:])
            
            print(f"\n[SFT Step {sft_step:,}] Val Loss: {val_loss:.4f} | WRM: {avg_wrm:.4f} | Time: {elapsed:.2f}h")
            print(f"  WRM Breakdown: Wave={wrm_breakdown['wave_anchor']:.4f}, Byte={wrm_breakdown['byte_transition']:.4f}")
            
            if val_loss < best_sft_loss:
                best_sft_loss = val_loss
                save_model_checkpoint(model, f'Flux-LM-Universal-{MODEL_SCALE}-SFT-best.pt')
                print(f"  ✓ New best SFT model!")
        
        # Checkpoint + demo
        if sft_step % SFT_SAVE_EVERY == 0:
            ckpt_path = CKPT_DIR / f'flux_lm_sft_step{sft_step}.pt'
            save_sft_checkpoint(model, sft_optimizer, sft_scheduler, sft_step, loss_val, ckpt_path)
            sft_generation_demo(model, device, sft_step)

pbar.close()

# Final save
print("\n" + "=" * 60)
print("SFT Training Complete!")
print("=" * 60)

save_model_checkpoint(model, f'Flux-LM-Universal-{MODEL_SCALE}-SFT.pt')
sft_generation_demo(model, device, sft_step)

total_time = (time.time() - start_time) / 3600
avg_sft = sum(sft_losses[-100:]) / len(sft_losses[-100:])
avg_wrm = sum(wrm_losses[-100:]) / len(wrm_losses[-100:])

print(f"\nSFT Summary:")
print(f"  Total time: {total_time:.2f} hours")
print(f"  Final SFT loss: {sft_losses[-1]:.4f}")
print(f"  Final WRM loss: {wrm_losses[-1]:.4f}")
print(f"  Best val loss: {best_sft_loss:.4f}")
print(f"\nWRM Anti-Forgetting Metrics:")
print(f"  Avg preservation loss: {avg_wrm:.4f}")

## Phase 2: DPO (Direct Preference Optimization)

After SFT, we use DPO to further align the model:

1. Generate multiple responses from FLUX-LM
2. Use a judge model (Qwen2.5-7B) to rank them
3. Train with DPO loss on preference pairs

**DPO Loss:**
$$\mathcal{L}_{DPO} = -\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)$$

In [ ]:
# Cell 14: DPO Configuration

# ==== DPO HYPERPARAMETERS ====
DPO_STEPS = 2000
DPO_LR = 5e-6  # Even lower than SFT
DPO_BETA = 0.1  # KL regularization strength
DPO_WARMUP_STEPS = 100
DPO_SAMPLES = 500  # Preference pairs to generate

print(f"DPO Configuration:")
print(f"  Steps: {DPO_STEPS}")
print(f"  Learning rate: {DPO_LR}")
print(f"  Beta: {DPO_BETA}")
print(f"  Preference pairs: {DPO_SAMPLES}")

In [ ]:
# Cell 15: Generate DPO Preference Pairs

DPO_DATA_PATH = CKPT_DIR / 'dpo_data.json'

def generate_candidates(model, prompt: str, n: int = 4) -> List[str]:
    """Generate multiple candidates from FLUX-LM."""
    candidates = []
    model.eval()
    
    for _ in range(n):
        try:
            output = model.generate(prompt, GenerationConfigUniversal(
                max_new_bytes=200,
                temperature=0.9,  # Higher temp for diversity
                top_p=0.95,
            ))
            candidates.append(output[len(prompt):])
        except:
            continue
    
    model.train()
    return candidates


def judge_pair(judge, tokenizer, prompt: str, resp_a: str, resp_b: str) -> str:
    """Use judge model to pick better response."""
    judge_prompt = f"""Compare these two responses to the question. Which is better? Answer with just "A" or "B".

Question: {prompt}

Response A: {resp_a[:300]}

Response B: {resp_b[:300]}

Better response:"""
    
    inputs = tokenizer(judge_prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(judge.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = judge.generate(**inputs, max_new_tokens=1, pad_token_id=tokenizer.pad_token_id)
    
    choice = tokenizer.decode(outputs[0][-1:], skip_special_tokens=True).strip()
    return choice if choice in ['A', 'B'] else random.choice(['A', 'B'])


# Check for cached DPO data
if DPO_DATA_PATH.exists():
    print(f"Loading cached DPO data from {DPO_DATA_PATH}")
    with open(DPO_DATA_PATH, 'r') as f:
        dpo_data = json.load(f)
    print(f"✓ Loaded {len(dpo_data)} preference pairs")
else:
    print(f"Generating {DPO_SAMPLES} DPO preference pairs...")
    
    # Load judge model
    print("\nLoading judge model (Qwen2.5-7B-Instruct)...")
    judge, judge_tokenizer = load_teacher('assistant')  # Qwen2.5-7B-Instruct
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
    
    dpo_data = []
    prompts_for_dpo = [
        "<|user|>What is the capital of France?<|end|>\n<|assistant|>",
        "<|user|>Explain machine learning briefly.<|end|>\n<|assistant|>",
        "<|problem|>\nWhat is 15% of 80?\n<|end|>\n<|reasoning|>\n",
        "<|lang:python|><|code|>\ndef factorial(n):\n",
    ]
    
    for i in tqdm(range(min(DPO_SAMPLES, 50)), desc="Generating DPO pairs"):
        prompt = prompts_for_dpo[i % len(prompts_for_dpo)]
        
        # Generate candidates from FLUX-LM
        candidates = generate_candidates(model, prompt, n=4)
        if len(candidates) < 2:
            continue
        
        # Judge to pick best and worst
        best_idx, worst_idx = 0, 1
        best_resp, worst_resp = candidates[0], candidates[1]
        
        for j in range(2, len(candidates)):
            choice = judge_pair(judge, judge_tokenizer, prompt, best_resp, candidates[j])
            if choice == 'B':
                worst_resp = best_resp
                best_resp = candidates[j]
        
        dpo_data.append({
            'prompt': prompt,
            'chosen': best_resp,
            'rejected': worst_resp,
        })
    
    # Unload judge
    unload_teacher(judge, judge_tokenizer)
    
    # Save DPO data
    with open(DPO_DATA_PATH, 'w') as f:
        json.dump(dpo_data, f, indent=2)
    print(f"\n✓ Saved {len(dpo_data)} preference pairs to {DPO_DATA_PATH}")
    
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        shutil.copy2(DPO_DATA_PATH, DRIVE_CKPT_DIR / 'dpo_data.json')
        print(f"✓ Copied to Google Drive")

print(f"\nSample preference pair:")
if dpo_data:
    print(f"  Prompt: {dpo_data[0]['prompt'][:50]}...")
    print(f"  Chosen: {dpo_data[0]['chosen'][:50]}...")
    print(f"  Rejected: {dpo_data[0]['rejected'][:50]}...")

In [ ]:
# Cell 16: DPO Training (Simplified)

# Note: Full DPO requires reference model. This is a simplified version.
# For production, use trl library's DPOTrainer.

print("=" * 60)
print("DPO Training (Simplified)")
print("=" * 60)

if len(dpo_data) < 10:
    print("⚠ Not enough DPO data. Skipping DPO phase.")
    print("  Generate more preference pairs or use existing SFT model.")
else:
    # Fresh optimizer for DPO
    dpo_optimizer = torch.optim.AdamW(model.parameters(), lr=DPO_LR, weight_decay=0.01)
    
    model.train()
    dpo_losses = []
    
    for step in tqdm(range(min(DPO_STEPS, len(dpo_data) * 10)), desc="DPO"):
        pair = dpo_data[step % len(dpo_data)]
        
        # Encode chosen and rejected
        chosen_text = pair['prompt'] + pair['chosen']
        rejected_text = pair['prompt'] + pair['rejected']
        
        chosen_ids = torch.tensor(encode_special(chosen_text)[:511], device=device).unsqueeze(0)
        rejected_ids = torch.tensor(encode_special(rejected_text)[:511], device=device).unsqueeze(0)
        
        # Simplified DPO: maximize log prob of chosen, minimize rejected
        with torch.amp.autocast('cuda'):
            chosen_out = model(chosen_ids[:, :-1], chosen_ids[:, 1:])
            rejected_out = model(rejected_ids[:, :-1], rejected_ids[:, 1:])
            
            # Preference loss: chosen should have lower loss than rejected
            dpo_loss = F.relu(chosen_out['loss'] - rejected_out['loss'] + 0.1)
        
        dpo_loss.backward()
        
        if (step + 1) % SFT_GRAD_ACCUM == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            dpo_optimizer.step()
            dpo_optimizer.zero_grad(set_to_none=True)
        
        dpo_losses.append(dpo_loss.item())
    
    print(f"\n✓ DPO complete. Final loss: {dpo_losses[-1]:.4f}")
    
    # Save DPO model
    save_model_checkpoint(model, f'Flux-LM-Universal-{MODEL_SCALE}-DPO.pt')
    sft_generation_demo(model, device, step=-1)

In [ ]:
# Cell 17: Final Evaluation

print("=" * 60)
print("Final Aligned Model Evaluation")
print("=" * 60)

model.eval()

eval_prompts = [
    ("assistant", "<|user|>What is machine learning?<|end|>\n<|assistant|>"),
    ("reasoning", "<|problem|>\nSarah has 15 apples. She gives 4 to Tom. How many does she have?\n<|end|>\n<|reasoning|>\n"),
    ("code", "<|lang:python|><|code|>\ndef fibonacci(n):\n    \"\"\"Return nth Fibonacci number.\"\"\"\n"),
]

for domain, prompt in eval_prompts:
    print(f"\n[{domain.upper()}]")
    print(f"Prompt: {prompt[:60]}...")
    
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=200,
        temperature=0.7 if domain != 'code' else 0.3,
        domain=domain,
    ))
    
    print(f"Response: {output[len(prompt):300]}")
    print("-" * 50)

In [ ]:
# Cell 18: Summary and Next Steps

print("\n" + "=" * 60)
print("SFT + DPO Training Complete!")
print("=" * 60)

print(f"\nCheckpoints saved:")
for ckpt in CKPT_DIR.glob(f'Flux-LM-Universal-{MODEL_SCALE}-*.pt'):
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  - {ckpt.name} ({size_mb:.1f} MB)")

if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    print(f"\n📁 Also on Google Drive: {DRIVE_CKPT_DIR}")

print(f"\nModel evolution:")
print(f"  1. Pre-trained: Flux-LM-Universal-{MODEL_SCALE}.pt")
print(f"  2. SFT: Flux-LM-Universal-{MODEL_SCALE}-SFT.pt")
print(f"  3. DPO: Flux-LM-Universal-{MODEL_SCALE}-DPO.pt (if DPO ran)")

print(f"\nNext steps:")
print(f"  1. Evaluate on benchmarks (MMLU, HumanEval, GSM8K)")
print(f"  2. Deploy for inference")
print(f"  3. Collect real user feedback for further alignment")